In [ ]:
import numpy as np, pandas as pd

df['bbox_w'] = df.bbox_x_br - df.bbox_x_tl
df['bbox_h'] = df.bbox_y_br - df.bbox_y_tl
df['bbox_area'] = df.bbox_w * df.bbox_h
df['aspect_ratio'] = df.bbox_w / df.bbox_h.replace(0, np.nan)
print(len(df), df.instance_label.nunique())

In [ ]:
c = df.instance_label.value_counts()
print('классов:', len(c))
print('<10 примеров:', (c < 10).sum())
print('<50 примеров:', (c < 50).sum())
print('<100 примеров:', (c < 100).sum())
print('доля боксов в топ-20 классах:', round(c.head(20).sum() / c.sum(), 3))
print('классов в предсказаниях:', ev._raw_preds_df.instance_label.nunique())
print(c.tail(20))

In [ ]:
df['z_max'] = np.nan
for cls, g in df.groupby('instance_label'):
    if len(g) < 15:
        continue
    zs = []
    for m in ['bbox_area', 'aspect_ratio']:
        v = np.log(g[m].where(g[m] > 0))
        med = v.median()
        mad = (v - med).abs().median()
        if mad > 0:
            zs.append((0.6745 * (v - med) / mad).abs())
    if zs:
        df.loc[g.index, 'z_max'] = pd.concat(zs, axis=1).max(axis=1)

for z in [3, 4, 5, 6, 8]:
    print(z, (df.z_max > z).sum())

In [ ]:
Z = 5
size_out = df[df.z_max > Z].sort_values('z_max', ascending=False)
print(len(size_out), round(100 * len(size_out) / len(df), 2), '%')
print(size_out.instance_label.value_counts().head(15))

In [ ]:
p = ev._raw_preds_df
p = p[p.confidence >= 0.6]
print('предсказаний после фильтра:', len(p))

m = df.merge(p, on='image_name', suffixes=('', '_p'))
print('пар:', len(m))

x1 = np.maximum(m.bbox_x_tl, m.bbox_x_tl_p)
y1 = np.maximum(m.bbox_y_tl, m.bbox_y_tl_p)
x2 = np.minimum(m.bbox_x_br, m.bbox_x_br_p)
y2 = np.minimum(m.bbox_y_br, m.bbox_y_br_p)
inter = (x2 - x1).clip(0) * (y2 - y1).clip(0)
a = (m.bbox_x_br - m.bbox_x_tl) * (m.bbox_y_br - m.bbox_y_tl)
b = (m.bbox_x_br_p - m.bbox_x_tl_p) * (m.bbox_y_br_p - m.bbox_y_tl_p)
m['iou'] = inter / (a + b - inter)

misclass = m[(m.iou >= 0.5) &
             (m.instance_label_p != m.instance_label) &
             m.instance_label.notna()].sort_values('confidence', ascending=False)
print('кандидатов:', len(misclass))

In [ ]:
cm = pd.crosstab(misclass.instance_label, misclass.instance_label_p)
pairs = cm.stack().sort_values(ascending=False)
print(pairs[pairs > 0].head(25))

In [ ]:
matched = set(zip(m[m.iou >= 0.5].image_name,
                  m[m.iou >= 0.5].bbox_x_tl_p, m[m.iou >= 0.5].bbox_y_tl_p))
p2 = p[~pd.Series(list(zip(p.image_name, p.bbox_x_tl, p.bbox_y_tl)),
                  index=p.index).isin(matched)]
missing = p2[p2.confidence >= 0.8].sort_values('confidence', ascending=False)
print('возможных пропусков:', len(missing))
print(missing.instance_label.value_counts().head(15))

In [ ]:
import os, matplotlib.pyplot as plt
from PIL import Image
ROOT = '/images'

def sheet(cand, n=50, cols=10, cap=None):
    sub = cand.head(n)
    rows = int(np.ceil(len(sub) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(1.6*cols, 1.9*rows))
    axes = np.atleast_2d(axes).reshape(-1)
    for ax, (_, r) in zip(axes, sub.iterrows()):
        try:
            im = Image.open(os.path.join(ROOT, r.image_name)).convert('RGB')
            W, H = im.size
            w, h = r.bbox_x_br - r.bbox_x_tl, r.bbox_y_br - r.bbox_y_tl
            ax.imshow(im.crop((max(0, r.bbox_x_tl - w*0.2), max(0, r.bbox_y_tl - h*0.2),
                               min(W, r.bbox_x_br + w*0.2), min(H, r.bbox_y_br + h*0.2)))
                        .resize((128, 128)))
        except Exception:
            ax.text(.5, .5, 'нет файла', ha='center', fontsize=6)
        ax.set_title(cap(r) if cap else str(r.instance_label), fontsize=6, pad=2)
        ax.axis('off')
    for ax in axes[len(sub):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

sheet(size_out, cap=lambda r: f'{r.instance_label}\nz={r.z_max:.1f}')
sheet(misclass, cap=lambda r: f'{r.instance_label}→\n{r.instance_label_p}')

In [ ]:
cols = ['task_id','task_name','image_name','instance_label',
        'bbox_x_tl','bbox_y_tl','bbox_x_br','bbox_y_br']

r1 = size_out[cols + ['bbox_w','bbox_h','z_max']].assign(тип='выброс по размеру')
r2 = misclass[cols + ['instance_label_p','confidence','iou']].assign(тип='возможно другой класс')

path = '/errors_report.xlsx'
with pd.ExcelWriter(path) as xl:
    r1.to_excel(xl, 'выбросы_размер', index=False)
    r2.to_excel(xl, 'неверный_класс', index=False)
    cm.to_excel(xl, 'матрица_классов')
    c.rename('боксов').to_frame().to_excel(xl, 'классы')
    pd.concat([r1, r2]).groupby(['тип','task_name']).size().rename('шт')\
      .reset_index().sort_values('шт', ascending=False).to_excel(xl, 'по_задачам', index=False)
print(path)